# Backfill AQI + Pha 1→2→3 trên Google Colab

Xem `WORKPLAN.md` § *HƯỚNG DẪN: Code nhẹ ở local (Mac) → chạy collect trên Google Colab* để hiểu bối cảnh.

**Chạy tuần tự từng cell.** Nếu Colab ngắt kết nối giữa chừng (free tier ~90 phút idle),
mở lại notebook và chạy lại từ Cell 1 — Cell 4 có `--resume` nên sẽ tiếp tục chỗ dở,
không gọi lại các call đã xong.

**Trước khi chạy:** thêm secret `OWM_API_KEY` ở panel 🔑 bên trái (Secrets), bật quyền
"Notebook access" cho secret đó.

## Cell 1 — Lấy code + cài deps

In [ ]:
!git clone -b claude/exciting-cannon-artc3j https://github.com/khuyn110400/big-data-aqi.git
%cd big-data-aqi
!pip install -q -r collector/requirements.txt
!pip install -q -r spark/requirements.txt

## Cell 1b — Java 17 cho Spark (bắt buộc cho Pha 1→2→3, không cần cho backfill)

Colab thường có sẵn OpenJDK 11, nhưng repo này dùng `pyspark==3.5.9` đã test với
Java 17 (xem WORKPLAN §1: ĐừNG dùng Java 21, lỗi reflection). Cài rõ ràng cho chắc.

In [ ]:
!apt-get install -qq openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!java -version

## Cell 2 — Mount Google Drive (nơi lưu dữ liệu, không mất khi Colab ngắt)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = "/content/drive/MyDrive/big-data-aqi/air-quality"
import os
os.makedirs(DATA, exist_ok=True)
print("Du lieu se luu vao:", DATA)

## Cell 3 — API key qua Colab Secrets (KHÔNG hard-code, KHÔNG commit key)

In [ ]:
import os
from google.colab import userdata
os.environ["OWM_API_KEY"] = userdata.get("OWM_API_KEY")
print("Da doc OWM_API_KEY tu Colab Secrets (do dai:", len(os.environ["OWM_API_KEY"]), "ky tu)")

## Cell 3b — Dry-run: xác nhận số call trước khi chạy thật (không tốn quota)

In [ ]:
!python collector/src/backfill_history.py \
    --cities collector/config/cities.json \
    --from 2021-01-01 --to 2026-09-01 \
    --out {DATA}/raw --dry-run

## Cell 4 — Chạy backfill thật vào Drive

~200 trạm x ~23 cửa sổ 90 ngày ≈ 4.600 call ≈ 1,3 giờ ở 1 call/giây. Có `--resume`:
chạy lại cell này sau khi bị ngắt sẽ tự bỏ qua các cửa sổ đã xong (đọc từ
`{DATA}/raw/_checkpoint.json` trên Drive).

In [ ]:
!python collector/src/backfill_history.py \
    --cities collector/config/cities.json \
    --from 2021-01-01 --to 2026-09-01 \
    --out {DATA}/raw --resume

## Cell 5 — Pha 1 → 2 → 3 ngay trên dữ liệu Drive (chỉ đổi đường dẫn, logic không đổi)

In [ ]:
!PYTHONPATH=spark python spark/jobs/phase1_clean.py \
    --input {DATA}/raw --output {DATA}/clean

In [ ]:
!PYTHONPATH=spark python spark/jobs/phase2_aqi.py \
    --input {DATA}/clean --output {DATA}/aqi --comparison-output docs/

In [ ]:
!PYTHONPATH=spark python spark/jobs/phase3_aggregate.py \
    --input {DATA}/aqi --output {DATA}/agg

## Cell 6 — Báo cáo số liệu cho `docs/experiments.md §1`

In số bản ghi, dung lượng từng tầng để điền tay vào bảng trong `docs/experiments.md`
(phần "vì sao cần big data" của đồ án).

In [ ]:
import subprocess

def du(path):
    out = subprocess.run(["du", "-sh", path], capture_output=True, text=True).stdout
    return out.split()[0] if out else "?"

for layer in ("raw", "clean", "aqi", "agg"):
    p = f"{DATA}/{layer}"
    print(f"{layer:6s}: {du(p)}\t{p}")